### Imports and main params

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats
from mpl_toolkits.axes_grid1 import make_axes_locatable

import mne
from mne.channels import find_ch_adjacency
from mne.datasets import sample
from mne.stats import combine_adjacency, spatio_temporal_cluster_test
from mne.viz import plot_compare_evokeds

import os
import pickle
import pandas as pd
from itertools import combinations
from scipy.stats import ttest_ind
from mne.stats import f_mway_rm, f_threshold_mway_rm

mne.set_log_level('INFO')

In [ ]:
expt = 'EXPT'
ROOT = f'/path/to/{expt}'
os.chdir(ROOT)

epochs_dir = join(ROOT, 'data/epochs/')
evokeds_dir = join(ROOT, 'data/evokeds/')
subjects_dir = join(ROOT,'data/mri/')
raw_dir = join(ROOT, 'data/raw/')
meg_dir = join(ROOT, 'data/meg/')
log_dir  = join(ROOT, 'logs')
stc_dir = join(ROOT, 'data/stc/')

excluded = ['R0000']
subjects = [i[:5] for i in os.listdir(raw_dir) if i.startswith('R') and i[:5] not in excluded and not i.endswith(".fif")]

output_dir = os.path.join(ROOT, 'plots')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

colors = {
    'CONDITION1': 'tab:blue',
    'CONDITION2': 'tab:orange',
    'CONDITION3': 'tab:green',
    'CONDITION4': 'tab:red'
}

conditions = list(colors.keys())

# permutation test vars 
tmin = 0
tmax = 0.8
tail = 1
p_thresh = 0.05
n_permutations = 10000

### Import epochs and subset by pres_type

In [ ]:
# Read data
for i, subject in enumerate(subjects):
    print(subject)
    epoch_file = os.path.join(meg_dir, subject, f'{subject}_{expt}-baselinecorr-ica-epo.fif')
    epochs = mne.read_epochs(epoch_file)

    if i == 0: # initialize variables on the first subject only
        times = epochs.times # get the epochs times
        conditions = epochs.metadata.condition.unique()
        epochs_data = np.zeros((len(subjects), len(conditions), 157,  len(times))) # create container for all subj epoch data (shape n_subj, n_conditions, n_sensors, n_times)
        info = epochs[0].info
        
        print('times = ', times)
        print('conditions = ', conditions)
    
    for j, condition in enumerate(conditions):
        epochs_data[i, j] = epochs[epochs.metadata['condition'] == condition].get_data(copy=True).mean(axis=0) # for each subj, for each cond: get average over trials  (axis 0)
print(epochs_data.shape)

In [ ]:
average_waveforms = epochs_data.mean(axis=0)  # average across the subject axis

print(average_waveforms.shape)

evoked_dict = {}
for i, condition in enumerate(conditions):
    evoked_dict[condition] = mne.EvokedArray(average_waveforms[i], info, tmin=times[0])

mne.viz.plot_compare_evokeds(evoked_dict, combine='gfp', title='Evoked Response') #, colors = colors)

### Create X

In [ ]:
X = epochs_data.copy()
print(X.shape)

# extract time window of interest
idx_tmin = np.where(times == 0)[0][0]
idx_tmax = np.where(times == 0.8)[0][0]
X = X[:, :, :, idx_tmin : idx_tmax + 1]
print(X.shape)

# create array of times for the search window: needed to get cluster times later
search_window = np.arange(0, 801, 1)
print(search_window)

# transpose for permutation test later
X = X.transpose(0, 1, 3, 2)
print(X.shape)

# # average over space
# X = X.mean(axis=3)
# print(X.shape)

# # convert to a list of len = num_conditions, each element of the list is an array of shape (nsubj x ntimes)
X = [np.squeeze(x) for x in np.split(X, 4, axis=1)]
print(len(X))
print(X[0].shape)

In [ ]:
def stat_fun(*args):
    # Return only the F-values
    return f_mway_rm(np.swapaxes(args, 1, 0), factor_levels=factor_levels,
                     effects=effects, return_pvals=False)[0]

### if doing spatio-temp search, add these vars + other modifs
adjacency, ch_names = mne.channels.find_ch_adjacency(info, ch_type='mag')
hemi = 'lh'

In [ ]:
# Estimating F-test threshold
factor_levels = [2,2,2] # shape of the condition contrast (3 factors, 2 levels each)
effects = ['A:B:C'] # A - main effect of A, B - main effect of B, A:B - interaction effect
f_thresh = f_threshold_mway_rm(len(subjects), factor_levels, effects, p_thresh)

print("Conditions: ", conditions)
print("Time window of analysis: ", tmin, " to ", tmax)
print("Launching clustering test for effect:", effects)
print("Threshold:", f_thresh)

# Perform the clustering test
F_obs, clusters, clusters_pvals, h0 = clu = mne.stats.permutation_cluster_test(X, #permutation_cluster_test
                                            tail=tail,                                   
                                            threshold=f_thresh,
                                            stat_fun = stat_fun,
                                            n_permutations=n_permutations,
                                            adjacency = adjacency,
                                            buffer_size=None,
                                            out_type='indices',
                                            n_jobs=-1)

# Check outputs
print("Cluster p-values:", clusters_pvals)
for cluster, pval in zip(clusters, clusters_pvals):
    #print(cluster)
    if pval < 0.3:
        cluster_start_time = search_window[cluster[0][0]]
        cluster_end_time = search_window[cluster[0][-1]]
        if len(search_window[cluster[0]]) > 3:
            print("%s - %s ms, p-value: %s" % (cluster_start_time, cluster_end_time, pval))

---

### plotting code

In [ ]:
evoked_dict = {}
for i, condition in enumerate(conditions):
    evoked_dict[condition] = mne.EvokedArray(average_waveforms[i], info, tmin=0)

# break down compare evokeds plot
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
mne.viz.plot_compare_evokeds(evoked_dict, combine='gfp', title='', axes = ax, show=False, colors=colors)
for cluster, pval in zip(clusters, clusters_pvals):
    if pval < 0.8:
        cluster_start_time = search_window[cluster[0][0]]
        cluster_end_time = search_window[cluster[0][-1]]
        if len(search_window[cluster[0]]) > 1:
            ax.axvspan(cluster_start_time, cluster_end_time,color='lightblue', alpha=0.5)
            print("%s - %s ms, p-value: %s" % (cluster_start_time, cluster_end_time, pval))

plt.show()